# Profitable strategies & modelling

### This notebook aims to study different investment strategies to assess which is more profitable. I will also include additional rules for muslim investors to remain Sharia-complient. This also aims at seeing ways to model a market.

In [1]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

from data import download_prices, compute_returns
from universe import get_universe, universe_diff, get_full_universe
from metrics import sharpe_ratio, annualised_return, max_drawdown, summary
from volatility import (
    rolling_vol, ewma_vol,
    fit_garch_universe, garch_conditional_vol,
    compare_universes
)
from signals import cross_section_zscore, momentum, short_term_reversal, volatility_signal

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 120
print('Imports OK')

Imports OK


In [2]:
sp500_tickers = get_full_universe('sp500')
spus_tickers  = get_full_universe('spus')

all_tickers  = list(set(sp500_tickers + spus_tickers))
prices_all   = download_prices(all_tickers, start='2015-01-01')

sp500_cols   = [t for t in sp500_tickers if t in prices_all.columns]
spus_cols    = [t for t in spus_tickers  if t in prices_all.columns]

prices_sp500 = prices_all[sp500_cols]
prices_spus  = prices_all[spus_cols]
returns_sp500 = compute_returns(prices_all[sp500_cols])
returns_spus  = compute_returns(prices_all[spus_cols])

print(f'SP500: {returns_sp500.shape[1]} stocks | SPUS: {returns_spus.shape[1]} stocks')

Fetched 503 S&P 500 tickers from Wikipedia
Fetched 503 S&P 500 tickers from Wikipedia
SPUS: 422 stocks (excluded 81)
Loading from cache: prices_2015-01-01_today.csv
SP500: 470 stocks | SPUS: 392 stocks


In [3]:
# Define columns in SP500 not in SPUS
columns_not_in_both = set(prices_sp500.columns) ^ set(prices_spus.columns)

prices_diff = pd.concat(
    [prices_sp500.reindex(columns=columns_not_in_both),
     prices_spus.reindex(columns=columns_not_in_both)],
    axis=0
)

returns_diff = pd.concat(
    [returns_sp500.reindex(columns=columns_not_in_both),
     returns_spus.reindex(columns=columns_not_in_both)],
    axis=0
)
columns_not_in_both = np.array(list(columns_not_in_both))